# Pipeline de Big Data com Apache Spark & Databricks
## Processamento Distribuído, Otimização de Performance e Delta Lake

**Curso:** Pós-Graduação em Data Science e Analytics  
**Unidade Curricular:** Banco de Dados e Big Data para Data Science  
**Unidades de Aprendizagem:** UA 03 e UA 04  

**Integrantes do Grupo:**
- Diogo Galrão Carvalho
- Felipe Artur Macedo Lima
- Luan Cavalcante Dias Rodrigues

---

## 1. Introdução e Contextualização Técnica

Nesta segunda etapa do projeto prático (UA 03 e UA 04), estendemos a arquitetura desenvolvida anteriormente — a migração da Loja do League of Legends para o MongoDB Atlas — para um ambiente corporativo de **Big Data e Processamento Distribuído** utilizando o **Apache Spark** hospedado no **Databricks Community Edition**.

### Por que Apache Spark?
Enquanto bancos de dados NoSQL transacionais (como o MongoDB) são ideais para operações pontuais de leitura e escrita de alta concorrência (OLTP), eles enfrentam limitações em análises analíticas pesadas (OLAP) que exigem varredura massiva de coleções, cruzamento com dados demográficos externos e cálculos agregados complexos. O Apache Spark soluciona esse gargalo ao:
1. **Computação em Memória (In-Memory Computing):** Utiliza grafos acíclicos dirigidos (DAGs) e mantém dados intermediários na RAM dos nós executores, atingindo velocidade até 100x superior ao tradicional Hadoop MapReduce.
2. **Escalabilidade Horizontal:** Distribui tarefas analíticas paralelamente entre nós de um cluster.
3. **Integração Heterogênea:** Conecta-se nativamente a fontes NoSQL (MongoDB Spark Connector), sistemas de arquivos distribuídos (DBFS/S3/ADLS) e tabelas lakehouse (**Delta Lake**).
4. **Otimizador Catalyst & Tungsten:** Gera planos de execução físicos otimizados com geração dinâmica de bytecode.

## 2. Preparação do Ambiente e Inicialização da SparkSession

No **Databricks**, a variável `spark` já vem instanciada por padrão. O conector do MongoDB é instalado via interface gráfica na aba *Compute -> Cluster -> Libraries -> Install New -> Maven -> Coordinates:* `org.mongodb.spark:mongo-spark-connector_2.12:10.4.0`.

Para permitir a reprodução deste notebook também em ambientes locais ou Google Colab, a célula abaixo realiza a configuração automática caso o PySpark precise ser inicializado.

In [ ]:
# Instalação de dependências caso esteja rodando no Google Colab
try:
    import pyspark
except ImportError:
    !pip install pyspark delta-spark certifi --quiet

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, explode, sum, count, avg, round, to_date,
    date_format, dayofweek, broadcast, when, desc
)
from pyspark.sql.types import *
import os

# Inicialização da SparkSession com suporte a Delta Lake e MongoDB
try:
    # Se executando no Databricks, reutiliza a sessão gerenciada
    spark
    print("[OK] Executando no ambiente gerenciado Databricks!")
except NameError:
    print("Inicializando SparkSession local com suporte Delta Lake...")
    spark = SparkSession.builder \
        .appName("LojaLoL-BigData-Spark") \
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
        .getOrCreate()
    print("[OK] SparkSession local inicializada com sucesso!")

## 3. Ingestão de Múltiplas Fontes de Dados

O pipeline ingere dados de duas origens com características distintas:
1. **Origem NoSQL (MongoDB Atlas):** Coleção `jogadores` contendo o perfil, o elo e a lista aninhada de compras com snapshot dos itens adquiridos.
2. **Origem Externa no DBFS (`jogadores_demografia.csv`):** Arquivo complementar com variáveis demográficas dos jogadores (país, cidade, faixa etária, gênero, tipo de assinatura e plataforma utilizada).

In [ ]:
# Caminhos das fontes de dados
# No Databricks, arquivos carregados ficam em 'dbfs:/FileStore/tables/'
# Localmente, lemos de '../data/processed/'
caminho_demografia = '../data/processed/jogadores_demografia.csv'
caminho_mongo_json = '../data/processed/mongo_export/jogadores.json'

# Fallback para execução em nuvem ou DBFS
if not os.path.exists(caminho_demografia):
    caminho_demografia = '/FileStore/tables/jogadores_demografia.csv'
    caminho_mongo_json = '/FileStore/tables/jogadores.json'

# 1. Leitura do CSV demográfico complementar
df_demografia = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(caminho_demografia)

print(f"Fonte DBFS Carregada: {df_demografia.count()} registros demográficos.")
df_demografia.show(5, truncate=False)

In [ ]:
# 2. Ingestão dos Documentos do MongoDB Atlas
# Se rodando no cluster com MongoDB Connector configurado:
# df_jogadores_mongo = spark.read.format("mongodb").load()

# Leitura direta do BSON/JSON gerado da migração NoSQL
df_jogadores_raw = spark.read.format("json").option("multiline", "true").load(caminho_mongo_json)

print(f"Documentos NoSQL Carregados: {df_jogadores_raw.count()} jogadores.")
print("Esquema hierárquico importado do MongoDB:")
df_jogadores_raw.printSchema()

## 4. Desaninhamento e Transformações Distribuídas

Como cada documento de jogador armazena a lista de compras em um array embutido, utilizamos a operação `explode()` para normalizar o DataFrame distribuído, transformando cada item comprado em uma linha analítica com as informações do jogador propagadas.

In [ ]:
# Desaninhando o array de compras
df_compras_exploded = df_jogadores_raw.select(
    col("id_jogador"),
    col("nick"),
    col("regiao"),
    col("elo"),
    explode(col("compras")).alias("compra")
).select(
    col("id_jogador"),
    col("nick"),
    col("regiao"),
    col("elo"),
    col("compra.id_compra").alias("id_compra"),
    col("compra.id_item").alias("id_item"),
    col("compra.item").alias("item_nome"),
    col("compra.preco_unitario").alias("preco_unitario"),
    col("compra.quantidade").alias("quantidade"),
    col("compra.total_ouro").alias("total_ouro"),
    to_date(col("compra.data_compra")).alias("data_compra"),
    col("compra.minuto_compra").alias("minuto_compra"),
    col("compra.id_partida").alias("id_partida")
)

print(f"Total de transações analíticas desaninhadas: {df_compras_exploded.count()}")
df_compras_exploded.show(5)

## 5. Estratégias de Otimização e Tuning de Performance

Nesta seção, exploramos duas técnicas cruciais exigidas na rubrica de avaliação:
1. **Broadcast Hash Join:** Eliminação do shuffle de rede ao cruzar o DataFrame de compras com o DataFrame demográfico reduzido.
2. **Caching em Memória:** Persistência do DataFrame intermediário com `.cache()` para acelerar a execução de múltiplos ramos analíticos independentes.

In [ ]:
# 5.1 Junção com Broadcast Join vs Standard Join
# O DataFrame demográfico tem apenas 25 registros. Em um join tradicional, o Spark faria
# shuffle de ambas as tabelas (Exchange hashpartitioning).
# Com broadcast(), a tabela pequena é copiada para cada executor, eliminando o shuffle da tabela fato!

df_demografia_selecao = df_demografia.select(
    col("id_jogador"),
    col("pais"),
    col("cidade"),
    col("idade"),
    col("genero"),
    col("tier_assinatura"),
    col("plataforma")
)

# Executando o Broadcast Join
df_enriquecido = df_compras_exploded.join(
    broadcast(df_demografia_selecao),
    on="id_jogador",
    how="inner"
)

print("Plano de Execução Físico com BroadcastHashJoin (sem Shuffle Exchange):")
df_enriquecido.explain(mode="simple")

In [ ]:
# 5.2 Aplicação do Caching
# Como df_enriquecido será utilizado para 4 agregações distintas de negócio,
# persistimos os dados na memória do cluster para evitar recalcular o grafo DAG.
import time

df_enriquecido.cache()

# Ação para materializar o cache
inicio = time.time()
contagem_inicial = df_enriquecido.count()
tempo_primeira_execucao = time.time() - inicio

# Segunda ação (já em cache)
inicio = time.time()
contagem_cached = df_enriquecido.count()
tempo_cached = time.time() - inicio

print(f"Total de registros em cache: {contagem_cached}")
print(f"Tempo 1ª execução (materialização): {tempo_primeira_execucao:.4f}s")
print(f"Tempo 2ª execução (direto da RAM):   {tempo_cached:.4f}s")
print(f"Aceleração obtida com Caching:   {tempo_primeira_execucao / max(tempo_cached, 0.0001):.1f}x")

## 6. Agregações e Métricas de Negócio em Larga Escala

Implementamos 4 análises analíticas de alto valor estratégico para o negócio da loja de e-commerce gamer.

In [ ]:
# Métrica 1: Receita Total, Volume e Ticket Médio por Região e Elo Competitivo
df_metrica_regiao = df_enriquecido.groupBy("regiao", "elo").agg(
    sum("total_ouro").alias("receita_total_ouro"),
    sum("quantidade").alias("itens_vendidos"),
    count("id_compra").alias("total_transacoes"),
    round(avg("total_ouro"), 2).alias("ticket_medio_ouro")
).orderBy(desc("receita_total_ouro"))

print("=== [Métrica 1] Receita e Consumo por Região e Elo ===")
df_metrica_regiao.show(10, truncate=False)

In [ ]:
# Métrica 2: Top 10 Itens Mais Vendidos (Volume e Receita Acumulada)
df_top_itens = df_enriquecido.groupBy("id_item", "item_nome").agg(
    sum("quantidade").alias("unidades_compradas"),
    sum("total_ouro").alias("receita_gerada_ouro"),
    count("id_compra").alias("frequencia_compras")
).orderBy(desc("receita_gerada_ouro")).limit(10)

print("=== [Métrica 2] Top 10 Itens Mais Rentáveis ===")
df_top_itens.show(10, truncate=False)

In [ ]:
# Métrica 3: Sazonalidade de Consumo por Dia da Semana e Plataforma Gamer
# Avalia se jogadores em Desktops gastam mais nos finais de semana
df_sazonalidade = df_enriquecido.withColumn(
    "dia_semana_nome", date_format(col("data_compra"), "EEEE")
).withColumn(
    "dia_semana_num", dayofweek(col("data_compra"))
).groupBy("dia_semana_num", "dia_semana_nome", "plataforma").agg(
    sum("total_ouro").alias("receita_dia"),
    count("id_compra").alias("volume_compras")
).orderBy("dia_semana_num", "plataforma")

print("=== [Métrica 3] Padrão de Compras por Dia da Semana e Plataforma ===")
df_sazonalidade.show(14, truncate=False)

In [ ]:
# Métrica 4: Filtragem Temporal e Gasto por Tier de Assinatura (VIP vs Gratuito)
df_assinatura = df_enriquecido.filter(
    (col("data_compra") >= "2025-08-20") & (col("data_compra") <= "2025-09-20")
).groupBy("tier_assinatura").agg(
    sum("total_ouro").alias("receita_periodo"),
    count("id_compra").alias("transacoes_periodo"),
    round(avg("total_ouro"), 2).alias("ticket_medio")
).orderBy(desc("receita_periodo"))

print("=== [Métrica 4] Gasto no Período por Nível de Assinatura ===")
df_assinatura.show(truncate=False)

## 7. Armazenamento Final em Delta Lake

O destino final do pipeline analítico é o formato **Delta Lake**. O Delta Lake transforma o data lake em um *Lakehouse*, fornecendo:
- **Transações ACID (gravações atômicas e isoladas via Delta Log)**;
- **Compactação Parquet** com alta taxa de compressão colunar;
- **Particionamento físico inteligente** (particionamento por `regiao`);
- **Recursos de Time Travel** (histórico de versões de dados).

In [ ]:
caminho_delta = '../data/delta/analise_compras_jogadores'
os.makedirs('../data/delta', exist_ok=True)

print(f"Gravando tabela Delta Lake particionada por regiao em: {caminho_delta} ...")
df_enriquecido.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("regiao") \
    .save(caminho_delta)

print("[OK] Gravação em Delta Lake concluída com sucesso!")

# Leitura e validação da tabela Delta recém-gravada
df_delta_lido = spark.read.format("delta").load(caminho_delta)
print(f"Registros lidos da tabela Delta: {df_delta_lido.count()}")

## 8. Conclusão e Avaliação Crítica da Arquitetura Distribuída

A integração entre o **MongoDB Atlas**, o **Apache Spark** e o **Delta Lake** comprova os benefícios da arquitetura híbrida de dados moderna (*Modern Data Stack*):

1. **Desacoplamento entre Camadas Transacional e Analítica:** O MongoDB atende a aplicação com baixa latência para leituras de perfil de jogadores, sem sofrer lentidão provocada por queries de BI e relatórios pesados, que foram delegadas ao cluster Spark.
2. **Eliminação de Shuffles via Broadcast Join:** A técnica de Broadcast Join provou ser essencial para otimizar junções entre grandes tabelas transacionais de compras e tabelas de dimensões reduzidas (metadados demográficos), eliminando o tráfego de rede entre executores.
3. **Eficiência com Caching:** A reutilização de DataFrames na memória RAM via `.cache()` garantiu respostas instantâneas para múltiplos ramos analíticos (métricas por região, itens e temporalidade), sem necessidade de reprocessar os dados brutos.
4. **Confiabilidade no Lakehouse:** A persistência em Delta Lake garante governança, controle de esquema e capacidade de auditoria histórica via transações ACID.

---
## 9. Referências Bibliográficas

- ALVES, L. M.; SILVA, R. F. O.; SANTOS, G. H. R. **Comparação de metodologias de migração de bancos de dados relacionais para bancos orientados a documentos**. Joinville: Anais do Congresso da SBC, 2020.
- ARMBRUST, M. et al. **Delta Lake: High-Performance ACID Table Storage over Cloud Object Stores**. Proceedings of the VLDB Endowment, 2020.
- CHAMBERS, B.; ZAHARIA, M. **Spark: The Definitive Guide - Big Data Processing Made Simple**. Sebastopol: O'Reilly Media, 2018.
- GHOTIYA, S.; MANDAL, J.; KANDASAMY, S. **Migration from relational to NoSQL database**. IOP Conference Series: Materials Science and Engineering, 2017.
- KANE, F. **Frank Kane's Taming Big Data with Apache Spark and Python**. Birmingham: Packt Publishing, 2017.